# Re-run: reliability of the two formats (RQ1)

This notebook recomputes the point estimates of Table 1 and the per-tuple ICC from the per-case file
published with release `checklist-grading-2026`, and nothing else: no model call, no corpus. It needs
Python 3.9+ with numpy, and `per-case-model-outputs.jsonl` beside this notebook (or in `RELEASE_DIR`).
The 95% intervals come from a 4,000-replicate cluster bootstrap in the release's code package; this
notebook checks the point estimates.

In [1]:
# Finding the published data. This cell is the same in every notebook of this report.
import gzip, json, os, sys
from pathlib import Path

# Importing from the release's code/ folder must not leave __pycache__ folders in it.
sys.dont_write_bytecode = True

RELEASE_ID = "checklist-grading-2026"   # set when the report is built
DATA_NAME = "per-case-model-outputs.jsonl"


def find_release_file(name=DATA_NAME):
    """Look beside this notebook, in RELEASE_DIR, in the folders above, and in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    places += [here, *here.parents]
    places += [p / "research/wiki/research/releases" / RELEASE_ID for p in [here, *here.parents]]
    for d in places:
        for n in (name, name + ".gz"):
            if (d / n).is_file():
                return d / n
    raise FileNotFoundError(
        f"Could not find {name} (or {name}.gz). Put it in the same folder as this notebook: download it "
        f"from the report ('Download the data file', beside each notebook) or take it from the release "
        f"directory. Or set the environment variable RELEASE_DIR to the folder that holds it.")


def per_case(runs=None):
    """Yield the per-case records of the published file, optionally only for the named runs."""
    path = find_release_file()
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            if r.get("record") == "case" and (runs is None or r["run"] in runs):
                yield r


def code_dir():
    """The release's code package (its code/ folder): in RELEASE_DIR, beside this notebook or in a folder
    above it (as when the notebook sits in the unzipped release's notebooks/ folder), in a folder named
    after the release there, or in a repository checkout."""
    here = Path.cwd()
    places = [Path(os.environ["RELEASE_DIR"])] if os.environ.get("RELEASE_DIR") else []
    for p in [here, *here.parents]:
        places += [p, p / RELEASE_ID, p / "research/wiki/research/releases" / RELEASE_ID]
    for d in places:
        if (d / "code" / "SHA256SUMS").is_file():
            return d / "code"
    raise FileNotFoundError(
        f"Could not find the code package of release {RELEASE_ID} (its code/ folder). Download the release "
        f"from the report ('Download the release', in the sidebar or beside each notebook), unzip it, and "
        f"run this notebook from the unzipped folder or its notebooks/ folder. Or set the environment "
        f"variable RELEASE_DIR to the unzipped {RELEASE_ID} folder.")


def find_code_file(relpath):
    """A file inside the release's code package, by its path there (e.g. studies/.../out/result.json)."""
    path = code_dir() / relpath
    if not path.is_file():
        raise FileNotFoundError(f"The code package at {code_dir()} has no {relpath}: is it release {RELEASE_ID}?")
    return path


def load_code_json(relpath):
    """A committed output of the code package, parsed."""
    return json.loads(find_code_file(relpath).read_text(encoding="utf-8"))


def shown(path):
    """A path as printed: relative to this notebook's folder, so an executed notebook names no machine's folders."""
    return os.path.relpath(path, Path.cwd())


def corpus_file(env_var, name):
    """A public corpus file the reader has downloaded: $env_var/name, or name beside this notebook."""
    for d in ([Path(os.environ[env_var])] if os.environ.get(env_var) else []) + [Path.cwd()]:
        if (d / name).is_file():
            return d / name
    raise FileNotFoundError(f"This notebook also needs {name} from the public corpus. Download the corpus "
                            f"(its location and hash are in the release file), then set {env_var} to its "
                            f"folder or put {name} beside this notebook.")


print("Data:", shown(find_release_file()))

Data: per-case-model-outputs.jsonl


In [2]:
import json
import numpy as np

recs = [json.loads(l) for l in open(find_release_file(), encoding="utf-8")]
G = {(r["corpus"], r["id"], r["form"], r["model"], r["arm"], r["rep"]): r for r in recs if r["record"] == "grade"}
items = {(r["corpus"], r["id"]): r for r in recs if r["record"] == "item"}
MODELS, ARMS, REPS = ("gemini", "gpt"), ("likert", "boolean"), (0, 1)
DIMS = ["Correctness", "Code Elegance", "Readability", "Documentation"]

def val(corpus, i, m, a, k, dim=None):
    g = G.get((corpus, i, "orig", m, a, k))
    if g is None:
        return None
    return g["score"] if dim is None else (g["scores"] or {}).get(dim)

def complete(corpus, ids, dim=None):
    return [i for i in ids if all(val(corpus, i, m, a, k, dim) is not None for m in MODELS for a in ARMS for k in REPS)]

r = lambda x, y: float(np.corrcoef(x, y)[0, 1])
mohler = sorted(i for (c, i), it in items.items() if c == "mohler" and "main" in it["draws"])
men = sorted((i for (c, i) in items if c == "menagerie"), key=int)
print(len(mohler), "short answers;", len(men), "programs")

620 short answers; 272 programs


In [3]:
def table(corpus, ids, dim=None):
    ids = complete(corpus, ids, DIMS[0] if dim else None)
    x = {(m, a, k): np.array([val(corpus, i, m, a, k, dim) for i in ids]) for m in MODELS for a in ARMS for k in REPS}
    R = {(m, a): r(x[m, a, 0], x[m, a, 1]) for m in MODELS for a in ARMS}
    X = {a: r((x["gemini", a, 0] + x["gemini", a, 1]) / 2, (x["gpt", a, 0] + x["gpt", a, 1]) / 2) for a in ARMS}
    dR = np.mean([R[m, "boolean"] - R[m, "likert"] for m in MODELS])
    return len(ids), R, X, dR, X["boolean"] - X["likert"]

rows = {"Mohler short answers": table("mohler", mohler)}
for d in DIMS:
    rows[f"Menagerie {d}"] = table("menagerie", men, d)
for name, (n, R, X, dR, dX) in rows.items():
    print(f"{name:24s} n={n}  retest L/B gemini {R['gemini','likert']:.3f}/{R['gemini','boolean']:.3f}  "
          f"gpt {R['gpt','likert']:.3f}/{R['gpt','boolean']:.3f}  dR {dR:+.3f}  cross L/B {X['likert']:.3f}/{X['boolean']:.3f}  dX {dX:+.3f}")

Mohler short answers     n=620  retest L/B gemini 0.987/0.961  gpt 0.962/0.965  dR -0.011  cross L/B 0.885/0.906  dX +0.021
Menagerie Correctness    n=226  retest L/B gemini 0.707/0.596  gpt 0.663/0.426  dR -0.174  cross L/B -0.164/-0.159  dX +0.006
Menagerie Code Elegance  n=226  retest L/B gemini 0.830/0.655  gpt 0.673/0.562  dR -0.143  cross L/B 0.577/0.223  dX -0.354
Menagerie Readability    n=226  retest L/B gemini 0.832/0.747  gpt 0.745/0.445  dR -0.192  cross L/B 0.586/0.019  dX -0.567
Menagerie Documentation  n=226  retest L/B gemini 0.684/0.650  gpt 0.451/0.285  dR -0.100  cross L/B 0.231/0.212  dX -0.019


In [4]:
# The article's printed point estimates (Table 1).
PRINTED = {
    "Mohler short answers": ("0.987", "0.961", "0.962", "0.965", "-0.011", "0.885", "0.906", "+0.021"),
    "Menagerie Correctness": ("0.707", "0.596", "0.663", "0.426", "-0.174", "-0.164", "-0.159", "+0.006"),
    "Menagerie Code Elegance": ("0.830", "0.655", "0.673", "0.562", "-0.143", "0.577", "0.223", "-0.354"),
    "Menagerie Readability": ("0.832", "0.747", "0.745", "0.445", "-0.192", "0.586", "0.019", "-0.567"),
    "Menagerie Documentation": ("0.684", "0.650", "0.451", "0.285", "-0.100", "0.231", "0.212", "-0.019"),
}
for name, want in PRINTED.items():
    n, R, X, dR, dX = rows[name]
    got = (f"{R['gemini','likert']:.3f}", f"{R['gemini','boolean']:.3f}", f"{R['gpt','likert']:.3f}", f"{R['gpt','boolean']:.3f}",
           f"{dR:+.3f}", f"{X['likert']:.3f}", f"{X['boolean']:.3f}", f"{dX:+.3f}")
    assert got == want, (name, got, want)
print("Table 1's point estimates reproduce.")

Table 1's point estimates reproduce.


In [5]:
# Per-tuple ICC(3,1) between the two models (the health framework's unit) against response-level.
def icc31(Y):
    n, k = Y.shape
    gm = Y.mean()
    ssr = k * ((Y.mean(1) - gm) ** 2).sum()
    ssc = n * ((Y.mean(0) - gm) ** 2).sum()
    sse = ((Y - gm) ** 2).sum() - ssr - ssc
    msr, mse = ssr / (n - 1), sse / ((n - 1) * (k - 1))
    return (msr - mse) / (msr + (k - 1) * mse)

ids = complete("mohler", mohler)
tuples = []
for i in ids:
    for k in REPS:
        tuples += list(zip(G[("mohler", i, "orig", "gemini", "boolean", k)]["checks"],
                           G[("mohler", i, "orig", "gpt", "boolean", k)]["checks"]))
tup = icc31(np.array(tuples, float))
resp = {a: icc31(np.array([[val("mohler", i, "gemini", a, k), val("mohler", i, "gpt", a, k)] for k in REPS for i in ids]))
        for a in ARMS}
print(f"per (answer, check) tuple {tup:.3f}; per answer: checklist {resp['boolean']:.3f}, Likert {resp['likert']:.3f}")
assert (f"{tup:.3f}", f"{resp['boolean']:.3f}", f"{resp['likert']:.3f}") == ("0.803", "0.889", "0.865")
print("The per-tuple ICC reproduces.")

per (answer, check) tuple 0.803; per answer: checklist 0.889, Likert 0.865
The per-tuple ICC reproduces.
